# Day 16 — LLM Inference Systems：从 KV Cache 瓶颈到 Serving 优化

> Week 3 主线：LLM Inference → Agent → Graph × LLM

Day 15 建立了主线：

$$
\text{Autoregressive}
\rightarrow
\text{Prefill / Decode}
\rightarrow
\text{KV Cache}
\rightarrow
\text{Arithmetic Intensity}
\rightarrow
\text{Latency / Throughput}
$$

Day 16 继续追问：

> **既然 Decode 往往 memory-bound、KV cache 又会越来越大，系统和模型还能怎么优化？**

今天主线：

```text
KV cache bottleneck
↓
GQA / MQA / MLA
↓
Local / Sliding Window Attention
↓
Speculative Decoding
↓
Continuous Batching
↓
PagedAttention
↓
vLLM-style serving intuition
```

今天不要求实现真实 inference engine，也不要求安装 vLLM。目标是建立 serving systems 的设计直觉，并用小实验理解每种优化到底在解决什么瓶颈。

## 0. Day 15 快速热身

请先不翻前面笔记，回答：

1. 为什么 Prefill 往往比 Decode 更容易 compute-bound？
2. 为什么 KV cache 是 per-sequence 的？
3. 为什么 GQA 能减少 KV cache？
4. 为什么增大 batch 往往提高 throughput，但可能恶化 latency？
5. 为什么 KV cache memory 对 context length S 是 O(S)？

如果这 5 个问题基本能讲顺，再继续。

---
# 1. KV Cache Reduction：GQA / MQA / MLA

统一符号：

$$
Q:[B,T,N,H],\qquad K,V:[B,S,K,H]
$$

其中：

- $B$：batch / requests
- $T$：本次 query token 数
- $S$：历史 KV sequence length
- $N$：query heads
- $K$：KV heads
- $H$：head dimension

KV cache 粗略大小：

$$
\text{KV Memory}\propto B\times S\times L\times K\times H\times 2
$$

所以减少 $K$ 会直接减少 KV cache。

In [1]:
def kv_relative_size(num_query_heads, num_kv_heads):
    return num_kv_heads / num_query_heads

N = 40
for K in [40, 8, 4, 1]:
    ratio = kv_relative_size(N, K)
    print(f"query_heads={N:2d} | kv_heads={K:2d} | relative KV={ratio:.3f} | reduction≈{1/ratio:.1f}x")

query_heads=40 | kv_heads=40 | relative KV=1.000 | reduction≈1.0x
query_heads=40 | kv_heads= 8 | relative KV=0.200 | reduction≈5.0x
query_heads=40 | kv_heads= 4 | relative KV=0.100 | reduction≈10.0x
query_heads=40 | kv_heads= 1 | relative KV=0.025 | reduction≈40.0x


### 必须回答 1

MHA / GQA / MQA 的区别是什么？

请用一句核心逻辑回答：

> GQA / MQA 不是让不同 request 共享 KV，而是让同一个 quest 的不同 query head 共享一个 K/V。

## 1.2 MLA：不是“少几个 KV heads”，而是“压缩后再缓存”

GQA / MQA 的思路：

$$
\text{减少 } K
$$

MLA 的思路更像：

$$
h\rightarrow c=W_ch
$$

只长期缓存更小的 latent representation $c$，需要 attention 时再通过 learned projection 使用。

核心 trade-off：

```text
多一点固定模型参数 / projection
             ↕
大幅减少随 B、S 增长的动态 KV cache
```

注意：

- projection matrices 是 fixed model parameters；
- latent cache 是 per-token / per-sequence dynamic state；
- 真正持续膨胀的是随 $B\times S$ 增长的 cache。

### 必须回答 2

为什么额外存 W_c、W_Kc、W_Vc 仍然可能比直接缓存完整 KV 更省？

请明确区分：

- fixed model parameters
- dynamic per-token cache

回答：因为额外存的 $W_c$ , $W_Kc$ , $W_Vc$ 属于 fixed model parameters，所有 token 和 sequence 都共享，只需要存储一份，其内存不会随 batch size $B$ 和 context length $S$ 增长。
而完整 KV cache 属于 dynamic per-token/per-sequence state，会随着 $B\times S$ 持续增长。MLA 虽然增加了一些固定 projection parameters，但把每个 token 需要缓存的完整 K/V 压缩成低维 latent representation $c$，因此在长上下文和大 batch inference 中可以显著减少动态 cache memory。

---
# 2. Local / Sliding Window Attention：减少“要看的历史”

前面几种方法都在减少：

$$
\text{每个历史 token 要存多少 KV}
$$

另一个方向是减少：

$$
\text{每个新 Query 要读取多少历史 token}
$$

Full attention：

$$
\text{new query attends to } S \text{ history tokens}
$$

Sliding window：

$$
\text{new query attends to only } W \text{ recent tokens},\quad W\ll S
$$

In [2]:
def attention_kv_reads(context_len, window=None):
    return context_len if window is None else min(context_len, window)

for S in [1024, 4096, 16384, 65536]:
    full = attention_kv_reads(S)
    win = attention_kv_reads(S, window=4096)
    print(f"S={S:6d} | full_reads={full:6d} | window_reads={win:6d} | reduction≈{full/win:.1f}x")

S=  1024 | full_reads=  1024 | window_reads=  1024 | reduction≈1.0x
S=  4096 | full_reads=  4096 | window_reads=  4096 | reduction≈1.0x
S= 16384 | full_reads= 16384 | window_reads=  4096 | reduction≈4.0x
S= 65536 | full_reads= 65536 | window_reads=  4096 | reduction≈16.0x


### 必须回答 3

为什么 sliding window attention 可以降低 decode memory traffic？

如果窗口固定为 W，当 context length S 越来越大时，每一步需要读取的历史 KV 数量大致如何变化？

回答：Sliding Window Attention 只允许当前 Query 访问最近 W 个历史 token，因此 decode 时不需要读取完整长度为 S 的 KV cache。当 S>W 后，每一步需要读取的历史 KV 数量不再随 S 增长，而是大致固定为 W。因此可以显著减少 long-context inference 中的 HBM memory traffic。

---
# 3. Speculative Decoding：能不能一次“猜多个 token”？

Autoregressive 的限制：

```text
token t
↓
token t+1
↓
token t+2
```

Speculative decoding 的关键想法：

```text
小模型 / draft model
先快速猜多个 token
        ↓
大模型一次验证一段
        ↓
接受连续正确前缀
        ↓
不对的地方重新采样
```

它不是取消 autoregressive dependency，而是尝试减少昂贵 target model 的逐 token forward 次数。

In [3]:
draft = ["A", "B", "C", "D", "E"]
target_preferred = ["A", "B", "X", "Y", "Z"]

accepted = []
for d, t in zip(draft, target_preferred):
    if d == t:
        accepted.append(d)
    else:
        break

print("draft:", draft)
print("target preferred:", target_preferred)
print("accepted prefix:", accepted)
print("accepted length:", len(accepted))

draft: ['A', 'B', 'C', 'D', 'E']
target preferred: ['A', 'B', 'X', 'Y', 'Z']
accepted prefix: ['A', 'B']
accepted length: 2


### 必须回答 4

Speculative decoding 为什么不是简单地“让未来 token 并行生成”？

核心要点：

- draft 可以猜；
- target model 必须验证；
- 只有被验证接受的前缀才能真正进入最终序列。

回答：draft 一次先提出多个候选 token；target 利用这整个候选序列一次性并行计算各个位置的条件分布，然后按顺序决定哪些 draft token 可以接受。

---
# 4. Continuous Batching：为什么传统 fixed batch 不适合 LLM serving？

传统 batch 更像：

```text
凑一批请求
↓
一起开始
↓
最慢的请求结束
↓
整批结束
```

但 LLM request 的生成长度高度不一致。

Continuous batching 的核心：

> **每个 decode step 都允许已经完成的 request 离开，同时让新的 request 加入。**

In [4]:
requests = {"A": 3, "B": 7, "C": 2, "D": 5, "E": 4}
max_batch = 2
waiting = list(requests.items())
active = []
step = 0

while waiting or active:
    while len(active) < max_batch and waiting:
        name, remaining = waiting.pop(0)
        active.append([name, remaining])

    step += 1
    print(f"step {step:02d} active:", [(n, r) for n, r in active])

    for item in active:
        item[1] -= 1

    active = [item for item in active if item[1] > 0]

step 01 active: [('A', 3), ('B', 7)]
step 02 active: [('A', 2), ('B', 6)]
step 03 active: [('A', 1), ('B', 5)]
step 04 active: [('B', 4), ('C', 2)]
step 05 active: [('B', 3), ('C', 1)]
step 06 active: [('B', 2), ('D', 5)]
step 07 active: [('B', 1), ('D', 4)]
step 08 active: [('D', 3), ('E', 4)]
step 09 active: [('D', 2), ('E', 3)]
step 10 active: [('D', 1), ('E', 2)]
step 11 active: [('E', 1)]


### 必须回答 5

为什么 continuous batching 更适合 LLM serving？

请从这三个角度回答：

1. request 的生成长度不同；
2. request 到达时间不同；
3. GPU slot 不应该因为某个 request 提前结束而空着。

回答：LLM serving 中，每个 request 就是一次独立的用户生成任务。不同 request 的到达时间和生成长度不同，因此会在不同时间结束。Continuous batching 在每个 decode iteration 动态调度 batch：某个 request 完成后，立即移除它并加入等待中的新 request，从而避免 GPU slot 空闲，提高 GPU 利用率和整体 throughput。

---
# 5. PagedAttention：KV Cache 为什么也需要“操作系统式”管理？

如果每个 request 的 KV cache 都动态增长，而我们又要求它占一整块连续显存，就容易出现：

- 预留过多；
- memory fragmentation；
- 扩展不方便；
- request 结束后留下空洞。

PagedAttention 的核心直觉：

> **像虚拟内存分页一样，把 KV cache 拆成固定大小的 blocks/pages，让逻辑连续的 KV sequence 映射到物理上不连续的显存块。**

In [5]:
import math

def blocks_needed(tokens, block_size):
    return math.ceil(tokens / block_size)

block_size = 16
lengths = [7, 18, 33, 64, 70]

for s in lengths:
    blocks = blocks_needed(s, block_size)
    allocated = blocks * block_size
    waste = allocated - s
    print(f"tokens={s:3d} | blocks={blocks:2d} | allocated_slots={allocated:3d} | internal_waste={waste:2d}")

tokens=  7 | blocks= 1 | allocated_slots= 16 | internal_waste= 9
tokens= 18 | blocks= 2 | allocated_slots= 32 | internal_waste=14
tokens= 33 | blocks= 3 | allocated_slots= 48 | internal_waste=15
tokens= 64 | blocks= 4 | allocated_slots= 64 | internal_waste= 0
tokens= 70 | blocks= 5 | allocated_slots= 80 | internal_waste=10


### 必须回答 6

PagedAttention 主要在解决什么问题？

不要回答成“让 attention 变成分页计算”。

更准确地说，它主要优化的是：

> **KV cache 的 memory management / allocation。**

请解释 logical KV sequence、physical memory blocks，以及为什么不要求物理连续有好处。

回答：PagedAttention 主要优化 KV cache 的 memory management / allocation。它把一条 sequence 逻辑上连续的 KV cache 划分成固定大小的 logical blocks，再通过 block table 映射到物理显存中任意可用的 physical blocks，因此不要求整个 KV cache 在物理显存中连续。这样可以按需增长 KV cache，避免按最大生成长度提前预留大块显存，并显著降低 external fragmentation，提高显存利用率。

In [7]:
memory = [None] * 20

def alloc_contiguous(mem, name, size):
    for i in range(len(mem) - size + 1):
        if all(x is None for x in mem[i:i+size]):
            for j in range(i, i+size):
                mem[j] = name
            return True
    return False

def free(mem, name):
    for i, x in enumerate(mem):
        if x == name:
            mem[i] = None

alloc_contiguous(memory, "A", 5)
alloc_contiguous(memory, "B", 4)
alloc_contiguous(memory, "C", 5)
print("after A/B/C:", memory)

free(memory, "B")
print("after free B:", memory)

ok = alloc_contiguous(memory, "D", 6)
print("try allocate D(size=6):", ok)
print("memory:", memory)
print("free slots total:", sum(x is None for x in memory))

after A/B/C: ['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'C', 'C', 'C', 'C', 'C', None, None, None, None, None, None]
after free B: ['A', 'A', 'A', 'A', 'A', None, None, None, None, 'C', 'C', 'C', 'C', 'C', None, None, None, None, None, None]
try allocate D(size=6): True
memory: ['A', 'A', 'A', 'A', 'A', None, None, None, None, 'C', 'C', 'C', 'C', 'C', 'D', 'D', 'D', 'D', 'D', 'D']
free slots total: 4


### 必须回答 7

为什么“总空闲显存够”仍然可能无法为一个连续增长的 KV cache 分配空间？

这和 fragmentation 有什么关系？

回答：当显存存在 external fragmentation 时，空闲空间可能被分散成多个小块。即使这些小块的总容量大于某个 KV cache 所需空间，也可能不存在一块足够大的连续区域，因此传统 contiguous allocation 仍然会失败。PagedAttention 通过把 KV cache 划分为小 blocks，并允许这些 blocks 存放在非连续物理位置，从而利用这些零散空闲空间。

---
# 6. 把今天所有优化按“解决什么瓶颈”分类

| 方法 | 主要解决什么 | 核心动作 |
|---|---|---|
| GQA / MQA | KV cache 太大 | 减少 KV heads |
| MLA | KV cache 太大 | 压缩成 latent representation |
| Sliding Window | 历史 KV 读取太多 | 只看局部窗口 |
| Speculative Decoding | target model 逐 token decode 太慢 | draft 多步 + target 验证 |
| Continuous Batching | 动态请求导致 GPU 空槽 | 每步动态加入/移除请求 |
| PagedAttention | KV cache 分配/碎片化 | block/page 化管理 KV |

目标不是背名词，而是：

> **看到 bottleneck，就能猜到为什么会出现对应设计。**

### 必须回答 8

如果系统出现以下问题，你优先想到哪类方法？

1. KV cache 占满显存；
2. context 很长，每步都要读取大量历史 KV；
3. 请求生成长度差异很大，batch 中大量 slot 空闲；
4. 总空闲显存很多，但找不到足够连续空间；
5. 大模型 decode 每次只推进一个 token，想减少 target model forward 次数。

回答：
1. GQA / MQA / MLA：减少 KV cache footprint
2. Sliding Window / Sparse Attention：减少每步历史 KV reads
3. Continuous Batching：动态填充 GPU slots
4. PagedAttention：解决 KV cache allocation / fragmentation
5. Speculative Decoding：一次 target verification 推进多个候选 token

---
# 7. Serving Trade-off 小实验

下面只是极简 toy model：

- batch 越大，一轮可以服务更多 request；
- 但 batch 变大时单轮 latency 也上升；
- throughput = batch / latency。

In [8]:
def toy_latency(B, base=1.0, per_req=0.02):
    return base + per_req * B

for B in [1, 4, 8, 16, 32, 64, 128]:
    lat = toy_latency(B)
    thr = B / lat
    print(f"B={B:3d} | latency={lat:5.2f} | throughput={thr:6.2f}")

B=  1 | latency= 1.02 | throughput=  0.98
B=  4 | latency= 1.08 | throughput=  3.70
B=  8 | latency= 1.16 | throughput=  6.90
B= 16 | latency= 1.32 | throughput= 12.12
B= 32 | latency= 1.64 | throughput= 19.51
B= 64 | latency= 2.28 | throughput= 28.07
B=128 | latency= 3.56 | throughput= 35.96


### 必须回答 9

为什么上面的 latency 和 throughput 可以同时上升？

请从：

- 单轮处理更多 request；
- 单轮本身也更慢；
- throughput 衡量系统总产能；
- latency 衡量单请求/单轮等待时间；

这几个角度解释。

回答：随着 batch size 增大，一轮需要同时处理更多 request，因此单轮计算量增加、单轮耗时变长，所以单个 request 感受到的 latency 可能上升；但同一轮完成的 request 数量也增加，并且 GPU 利用率和权重读取的摊销通常更好，因此单位时间完成的总工作量增加，throughput 上升。二者并不矛盾，因为 latency 衡量单个请求的等待/完成时间，而 throughput 衡量整个系统单位时间的处理能力。

---
# 8. Day 16 最终复盘

请不翻答案，直接回答：

1. GQA / MQA 减少的到底是什么？
2. MLA 为什么“多一些 projection 参数”仍然可能非常划算？
3. Sliding window attention 减少的是哪一类 memory traffic？
4. Speculative decoding 为什么不是取消 autoregressive dependency？
5. Continuous batching 和 traditional fixed batching 的核心区别是什么？
6. PagedAttention 为什么更像 memory management，而不是新的 attention 公式？
7. 为什么 LLM serving workload 比普通静态 batch workload 更动态？
8. 哪些优化主要是 model-side，哪些主要是 system-side？
9. 如果只能用一句话总结 Day 16，你会怎么说？

建议一句话：

> LLM inference optimization 的核心不是“让模型多算”，而是 **减少不必要的 memory traffic、减少 sequential bottleneck，并更聪明地调度动态请求与 KV cache。**

# Day 16 完成标准

今天不要求：

- 安装 vLLM；
- 实现真实 PagedAttention kernel；
- 实现完整 speculative decoding；
- 背所有公式；
- 手搓 serving engine。

今天要求：

```text
能从 bottleneck 出发
↓
解释为什么需要某种优化
↓
知道它改变的是
compute / memory / scheduling / cache management
↓
能讲清 trade-off
```

如果能把下面这条线讲顺，Day 16 就完成：

```text
Decode memory-bound
↓
KV cache 太大 / 读得太多
↓
GQA / MQA / MLA / Sliding Window
↓
请求又是动态的
↓
Continuous Batching
↓
KV memory 还会碎片化
↓
PagedAttention
↓
进一步减少 target decode 次数
↓
Speculative Decoding
```

下一步 Day 17：

> **Agent / ReAct：从“token-by-token loop”走向“Thought / Action / Observation loop”。**